<a href="https://colab.research.google.com/github/alishbakhan21/pfdhfr-docking-resistance/blob/main/01_setup_and_structures.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Question: Can free docking tools reproduce a known drug pose in P. falciparum DHFR-TS, and do they show differences between the drug-sensitive (wild type) and drug-resistant (quadruple mutant) enzyme?

In [3]:
!pip install -q rdkit meeko gemmi
!apt-get -qq install -y openbabel > /dev/null
!wget -q https://github.com/ccsb-scripps/AutoDock-Vina/releases/download/v1.2.5/vina_1.2.5_linux_x86_64 -O vina
!chmod +x vina
!./vina --version

AutoDock Vina v1.2.5


In [5]:
!wget -q https://files.rcsb.org/download/1J3I.pdb
!wget -q https://files.rcsb.org/download/1J3K.pdb
!ls -lh *.pdb

-rw-r--r-- 1 root root 879K Sep 22  2025 1J3I.pdb
-rw-r--r-- 1 root root 912K Sep 22  2025 1J3K.pdb


In [7]:
import gemmi

for pdb in ["1J3I.pdb", "1J3K.pdb"]:
    st = gemmi.read_structure(pdb)
    print(pdb)
    for ch in st[0]:
        ligs = sorted({r.name for r in ch if r.het_flag == "H" and r.name != "HOH"})
        print("  chain", ch.name, "|", len(ch), "residues | ligands:", ligs)

1J3I.pdb
  chain A | 381 residues | ligands: ['NDP', 'WRA']
  chain B | 279 residues | ligands: ['NDP', 'WRA']
  chain C | 617 residues | ligands: ['UMP']
  chain D | 621 residues | ligands: ['UMP']
1J3K.pdb
  chain A | 461 residues | ligands: ['NDP', 'WRA']
  chain B | 325 residues | ligands: ['NDP', 'WRA']
  chain C | 755 residues | ligands: ['UMP']
  chain D | 806 residues | ligands: ['UMP']


Both structures contain the DHFR domain with WR99210 (WRA) and NADPH (NDP) bound in chains A and B, and the TS domain with dUMP in chains C and D. Residue counts include water molecules. For docking I'll use chain A only and keep NADPH, because the drug binds next to it. I'll remove water, dUMP and the other chains, and save the crystal pose of WR99210 to check my docking against.